# API Test Notebook
This notebook calls RF-DETR and Gemma4 APIs hosted on your server.

In [55]:
import requests
from pprint import pprint

RF_DETR_BASE = "http://129.97.170.141:8000"
GEMMA4_BASE = "http://129.97.170.141:8001"
IMAGE_URL = "https://i.ytimg.com/vi/DfcWOPpmw14/maxresdefault.jpg"
TIMEOUT = 120

In [56]:
# Health checks
rf_health = requests.get(f"{RF_DETR_BASE}/health", timeout=TIMEOUT).json()
gemma_health = requests.get(f"{GEMMA4_BASE}/health", timeout=TIMEOUT).json()

print("RF-DETR health:")
pprint(rf_health)
print()
print("Gemma4 health:")
pprint(gemma_health)

RF-DETR health:
{'cuda_visible_devices': '0', 'status': 'ok'}

Gemma4 health:
{'cuda_visible_devices': '1',
 'model_id': 'google/gemma-4-E2B-it',
 'status': 'ok'}


In [57]:
# RF-DETR POST /detect
detect_payload = {
    "image_url": IMAGE_URL,
    "threshold": 0.5,
    "save_annotated": False
}

detect_raw_response = requests.post(
    f"{RF_DETR_BASE}/detect",
    json=detect_payload,
    timeout=TIMEOUT,
)
detect_raw_response.raise_for_status()
detect_response = detect_raw_response.json()

print("RF-DETR detect response:")
pprint(detect_response)

RF-DETR detect response:
{'count': 3,
 'detections': [{'bbox_xyxy': [317.1911926269531,
                               68.65193176269531,
                               983.59375,
                               713.3673706054688],
                 'class_id': 1,
                 'class_name': 'person',
                 'confidence': 0.8328419327735901},
                {'bbox_xyxy': [903.4154052734375,
                               225.83653259277344,
                               1028.7303466796875,
                               352.5823669433594],
                 'class_id': 85,
                 'class_name': 'clock',
                 'confidence': 0.7544403076171875},
                {'bbox_xyxy': [825.169677734375,
                               1.2287360429763794,
                               961.77685546875,
                               133.96083068847656],
                 'class_id': 85,
                 'class_name': 'clock',
                 'confidence': 0.7157759666

In [58]:
# Gemma4 POST /generate with same image + RF-DETR output
generate_payload = {
    "prompt": "edit the RF-DETR detections json with 1 line description of each object as a new parameter (desc). look at the image and generate the descriptions. Keep it concise.",
    "image_url": IMAGE_URL,
    "rf_detr_output": detect_response,
    "max_new_tokens": 196
}

generate_raw_response = requests.post(
    f"{GEMMA4_BASE}/generate",
    json=generate_payload,
    timeout=TIMEOUT,
)
generate_raw_response.raise_for_status()
generate_response = generate_raw_response.json()

print("Gemma4 generate response:")
pprint(generate_response)

Gemma4 generate response:
{'response': {'content': '```json\n'
                         '[\n'
                         '  {"box_2d": [317, 68, 983, 961], "desc": "A person '
                         'in a patterned shirt"},\n'
                         '  {"box_2d": [903, 225, 1028, 352], "desc": "A clock '
                         'face"},\n'
                         '  {"box_2d": [825, 1, 961, 133], "desc": "A clock '
                         'face"}\n'
                         ']\n'
                         '```',
              'role': 'assistant'},
 'used_image_url': 'https://i.ytimg.com/vi/DfcWOPpmw14/maxresdefault.jpg'}


In [ ]:
# RF-DETR POST /detect_upload (direct bytes upload from app/camera capture)
LOCAL_IMAGE_PATH = "sample_frame.jpg"  # replace with a captured frame path

with open(LOCAL_IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

upload_raw_response = requests.post(
    f"{RF_DETR_BASE}/detect_upload",
    params={
        "threshold": 0.5,
        "save_annotated": "false",
        "output_path": "annotated_upload.jpg",
    },
    data=image_bytes,
    headers={"Content-Type": "image/jpeg"},
    timeout=TIMEOUT,
 )

upload_raw_response.raise_for_status()
upload_response = upload_raw_response.json()

print("RF-DETR detect_upload response:")
pprint(upload_response)